In [3]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "duckdb"])

0

In [5]:
!pip install duckdb --quiet

In [3]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas"])

0

In [5]:
import duckdb
import pandas as pd

con = duckdb.connect("hospital.duckdb")
print("DuckDB connected:", duckdb.__version__)

DuckDB connected: 1.4.4


In [7]:
con.execute("DROP TABLE IF EXISTS billing")
con.execute("DROP TABLE IF EXISTS visits")
con.execute("DROP TABLE IF EXISTS patients")

# -- Patients
con.execute("""
CREATE TABLE patients (
    patient_id          INTEGER  PRIMARY KEY,
    age                 INTEGER  NOT NULL CHECK (age BETWEEN 0 AND 120),
    gender              VARCHAR  NOT NULL,
    city                VARCHAR  NOT NULL,
    insurance_provider  VARCHAR  NOT NULL,
    chronic_flag        BOOLEAN  NOT NULL,
    registration_date   DATE     NOT NULL
)
""")

# -- Visits
con.execute("""
CREATE TABLE visits (
    visit_id              INTEGER     PRIMARY KEY,
    patient_id            INTEGER     NOT NULL,
    visit_date            DATE        NOT NULL,
    department            VARCHAR     NOT NULL,
    visit_type            VARCHAR     NOT NULL,
    length_of_stay_hours  DECIMAL(10,2) NOT NULL CHECK (length_of_stay_hours >= 0),
    risk_score            VARCHAR     NOT NULL CHECK (risk_score IN ('Low','Medium','High')),
    doctor_id             INTEGER     NOT NULL,
    FOREIGN KEY (patient_id) REFERENCES patients(patient_id)
)
""")

# -- Billing
con.execute("""
CREATE TABLE billing (
    bill_id         INTEGER      PRIMARY KEY,
    visit_id        INTEGER      NOT NULL UNIQUE,
    billed_amount   DECIMAL(12,2) NOT NULL CHECK (billed_amount >= 0),
    approved_amount DECIMAL(12,2),                -- nullable: 5.3% missing in data
    claim_status    VARCHAR      NOT NULL CHECK (claim_status IN ('Paid','Pending','Rejected')),
    payment_days    INTEGER,                       -- nullable: 3.2% missing in data
    billing_date    DATE         NOT NULL,
    FOREIGN KEY (visit_id) REFERENCES visits(visit_id)
)
""")

print("Tables created.")

Tables created.


In [9]:
con.execute("""
INSERT INTO patients
SELECT
    patient_id::INTEGER,
    age::INTEGER,
    gender::VARCHAR,
    city::VARCHAR,
    insurance_provider::VARCHAR,
    chronic_flag::BOOLEAN,
    registration_date::DATE
FROM read_csv_auto('patients.csv', header=true)
""")

con.execute("""
INSERT INTO visits
SELECT
    visit_id::INTEGER,
    patient_id::INTEGER,
    visit_date::DATE,
    department::VARCHAR,
    visit_type::VARCHAR,
    length_of_stay_hours::DECIMAL,
    risk_score::VARCHAR,
    doctor_id::INTEGER
FROM read_csv_auto('visits.csv', header=true)
""")

con.execute("""
INSERT INTO billing
SELECT
    bill_id::INTEGER,
    visit_id::INTEGER,
    billed_amount::DECIMAL,
    TRY_CAST(approved_amount AS DECIMAL),   -- handles nulls gracefully
    claim_status::VARCHAR,
    TRY_CAST(payment_days AS INTEGER),       -- handles nulls gracefully
    billing_date::DATE
FROM read_csv_auto('billing.csv', header=true)
""")

# Quick row count check
for t in ['patients','visits','billing']:
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"{t}: {n} rows")

patients: 5000 rows
visits: 25000 rows
billing: 25000 rows


In [11]:
indexes = [
    "CREATE INDEX IF NOT EXISTS idx_visits_visit_date     ON visits(visit_date)",
    "CREATE INDEX IF NOT EXISTS idx_visits_department     ON visits(department)",
    "CREATE INDEX IF NOT EXISTS idx_visits_risk_score     ON visits(risk_score)",
    "CREATE INDEX IF NOT EXISTS idx_visits_patient_id     ON visits(patient_id)",
    "CREATE INDEX IF NOT EXISTS idx_patients_city         ON patients(city)",
    "CREATE INDEX IF NOT EXISTS idx_patients_insurance    ON patients(insurance_provider)",
    "CREATE INDEX IF NOT EXISTS idx_billing_claim_status  ON billing(claim_status)",
    "CREATE INDEX IF NOT EXISTS idx_billing_billing_date  ON billing(billing_date)",
    "CREATE INDEX IF NOT EXISTS idx_billing_visit_id      ON billing(visit_id)",
]
for sql in indexes:
    con.execute(sql)
print("All indexes created.")


All indexes created.


## Why each index matters? 
| Index                        | Query it accelerates                           |
| ---------------------------- | ---------------------------------------------- |
| visits(visit_date)           | Time-based train/test splits in Phases 2–3     |
| visits(department)           | Operational KPIs — group by department         |
| visits(risk_score)           | High-risk % per department, doctor leaderboard |
| visits(patient_id)           | Star-join from patients to visits              |
| patients(insurance_provider) | Financial KPIs — rejection, realization        |
| billing(claim_status)        | Rejection rate, revenue leakage                |
| billing(billing_date)        | Time-series billing trends                     |

In [15]:
con.execute("""
CREATE OR REPLACE VIEW v_visit_enriched AS
SELECT
    v.visit_id, v.visit_date, v.department, v.visit_type,
    v.length_of_stay_hours, v.risk_score, v.doctor_id,
    p.patient_id, p.age, p.gender, p.city,
    p.insurance_provider, p.chronic_flag, p.registration_date
FROM visits v
JOIN patients p ON p.patient_id = v.patient_id
""")

con.execute("""
CREATE OR REPLACE VIEW v_finance_enriched AS
SELECT
    ve.visit_id, ve.visit_date, ve.department, ve.visit_type,
    ve.risk_score, ve.city, ve.insurance_provider,
    ve.age, ve.gender, ve.chronic_flag,
    b.bill_id, b.billed_amount, b.approved_amount,
    b.claim_status, b.payment_days, b.billing_date
FROM v_visit_enriched ve
JOIN billing b ON b.visit_id = ve.visit_id
""")

print("Views created.")

Views created.


In [19]:
# Q1: Top 10 departments by visit volume
q1 = con.execute("""
    SELECT department, COUNT(*) AS total_visits
    FROM visits
    GROUP BY department
    ORDER BY total_visits DESC
    LIMIT 10
""").fetchdf()
print("=== Q1: Top 10 departments by visit volume ===")
display(q1)

# Q2: Top 5 departments by avg length of stay
q2 = con.execute("""
    SELECT department, ROUND(AVG(length_of_stay_hours),2) AS avg_los_hours
    FROM visits
    GROUP BY department
    ORDER BY avg_los_hours DESC
    LIMIT 5
""").fetchdf()
print("=== Q2: Top 5 depts by avg length of stay ===")
display(q2)

# Q3: % High Risk visits per department
q3 = con.execute("""
    SELECT
        department,
        ROUND(100.0 * SUM(CASE WHEN risk_score='High' THEN 1 ELSE 0 END) / COUNT(*), 2)
            AS pct_high_risk
    FROM visits
    GROUP BY department
    ORDER BY pct_high_risk DESC
""").fetchdf()
print("=== Q3: % high risk visits per dept ===")
display(q3)

# Q4: Avg visits per patient by city
q4 = con.execute("""
    WITH vpp AS (
        SELECT patient_id, COUNT(*) AS visit_count
        FROM visits
        GROUP BY patient_id
    )
    SELECT p.city, ROUND(AVG(vpp.visit_count), 2) AS avg_visits_per_patient
    FROM patients p
    JOIN vpp ON vpp.patient_id = p.patient_id
    GROUP BY p.city
    ORDER BY avg_visits_per_patient DESC
""").fetchdf()
print("=== Q4: Avg visits per patient by city ===")
display(q4)

# Q5: Doctors handling most High Risk visits
q5 = con.execute("""
    SELECT doctor_id, COUNT(*) AS high_risk_visits
    FROM visits
    WHERE risk_score = 'High'
    GROUP BY doctor_id
    ORDER BY high_risk_visits DESC
    LIMIT 10
""").fetchdf()
print("=== Q5: Top doctors by high risk visits ===")
display(q5)

=== Q1: Top 10 departments by visit volume ===


,department,total_visits
0,General,4228
1,ER,4220
2,Neurology,4165
3,Orthopedics,4164
4,Cardiology,4159
5,ICU,4064


=== Q2: Top 5 depts by avg length of stay ===


,department,avg_los_hours
0,Neurology,19.72
1,Orthopedics,19.66
2,Cardiology,19.60
3,ER,19.53
4,General,19.43


=== Q3: % high risk visits per dept ===


,department,pct_high_risk
0,ICU,20.79
1,ER,20.66
2,Neurology,20.31
3,Orthopedics,20.22
4,General,19.84
5,Cardiology,18.99


=== Q4: Avg visits per patient by city ===


,city,avg_visits_per_patient
0,Pune,5.12
1,Hyderabad,5.06
2,Mumbai,5.02
3,Bangalore,5.02
4,Chennai,5.02
5,Delhi,4.95


=== Q5: Top doctors by high risk visits ===


,doctor_id,high_risk_visits
0,174,71
1,198,69
2,169,68
3,177,67
4,135,65
5,105,65
6,180,64
7,188,64
8,131,62
9,108,61


### Insights

#### Operational insights
1. Demand is concentrated but fairly balanced across major departments: General (4,228 visits) and ER (4,220) lead, with Neurology (4,165), Orthopedics (4,164), and Cardiology (4,159) close behind.

2. Average length of stay is highest in Neurology (~19.72 hours) and Orthopedics (~19.66 hours), which are natural first targets for throughput/bed-turnover improvement initiatives.

3. ICU has the highest share of High Risk visits (~20.79%), with ER next (~20.66%), suggesting ER–ICU pathways are key leverage points for high-risk triage and staffing.

4. Average visits per patient are highest in Pune (~5.08) and Hyderabad (~5.03), indicating higher repeat-demand (or chronic-care load) in those cities relative to others.

5. A small set of doctors handle the most High Risk visits (e.g., doctor_id 174: 71; 198: 69), which can inform targeted training, protocol reinforcement, and workload balancing for high-acuity care.

In [21]:
# Q6: Top 10 insurance providers by total billed
q6 = con.execute("""
    SELECT insurance_provider,
           ROUND(SUM(billed_amount),2) AS total_billed_amount
    FROM v_finance_enriched
    GROUP BY insurance_provider
    ORDER BY total_billed_amount DESC
    LIMIT 10
""").fetchdf()
print("=== Q6: Top insurance providers by total billed ===")
display(q6)

# Q7: Top 5 providers by claim rejection rate
q7 = con.execute("""
    SELECT
        insurance_provider,
        ROUND(100.0 * AVG(CASE WHEN claim_status='Rejected' THEN 1 ELSE 0 END), 2)
            AS rejection_rate_pct
    FROM v_finance_enriched
    GROUP BY insurance_provider
    ORDER BY rejection_rate_pct DESC
    LIMIT 5
""").fetchdf()
print("=== Q7: highest claim rejection rate by provider ===")
display(q7)

# Q8: Avg payment delay by provider
q8 = con.execute("""
    SELECT insurance_provider,
           ROUND(AVG(payment_days), 2) AS avg_payment_days
    FROM v_finance_enriched
    WHERE payment_days IS NOT NULL
    GROUP BY insurance_provider
    ORDER BY avg_payment_days DESC
""").fetchdf()
print("=== Q8: avg payment delay by provider ===")
display(q8)

# Q9: Revenue realization ratio by department
q9 = con.execute("""
    SELECT
        department,
        ROUND(AVG(approved_amount / NULLIF(billed_amount, 0)), 4) AS avg_realization_ratio
    FROM v_finance_enriched
    WHERE approved_amount IS NOT NULL
    GROUP BY department
    ORDER BY avg_realization_ratio ASC
""").fetchdf()
print("=== Q9: revenue realization ratio by department ===")
display(q9)

# Q10: High billed but approved = 0 or NULL (top 10% threshold)
q10 = con.execute("""
    WITH threshold AS (
        SELECT quantile_cont(billed_amount, 0.9) AS p90
        FROM billing
    )
    SELECT
        v.visit_id, v.department, p.insurance_provider,
        b.billed_amount, b.approved_amount, b.claim_status, b.payment_days
    FROM visits v
    JOIN patients p  ON p.patient_id = v.patient_id
    JOIN billing  b  ON b.visit_id   = v.visit_id
    CROSS JOIN threshold t
    WHERE b.billed_amount > t.p90
      AND (b.approved_amount IS NULL OR b.approved_amount = 0)
    ORDER BY b.billed_amount DESC
    LIMIT 20
""").fetchdf()
print("=== Q10: high billed but approved missing/Zero ===")
display(q10)


=== Q6: Top insurance providers by total billed ===


,insurance_provider,total_billed_amount
0,MediCareX,1.345912e+08
1,CareOne,1.307080e+08
2,HealthPlus,1.301807e+08
3,SecureLife,1.262890e+08


=== Q7: highest claim rejection rate by provider ===


,insurance_provider,rejection_rate_pct
0,SecureLife,15.69
1,MediCareX,15.25
2,HealthPlus,14.97
3,CareOne,14.87


=== Q8: avg payment delay by provider ===


,insurance_provider,avg_payment_days
0,SecureLife,13.08
1,HealthPlus,13.08
2,CareOne,13.03
3,MediCareX,13.01


=== Q9: revenue realization ratio by department ===


,department,avg_realization_ratio
0,Orthopedics,0.7685
1,Cardiology,0.7708
2,Neurology,0.7723
3,ER,0.7740
4,General,0.7747
5,ICU,0.7751


=== Q10: high billed but approved missing/Zero ===


,visit_id,department,insurance_provider,billed_amount,approved_amount,claim_status,payment_days
0,1570,General,HealthPlus,78054.79,NaN,Paid,8
1,18381,General,HealthPlus,68213.53,0.0,Rejected,9
2,15092,Neurology,CareOne,66410.33,NaN,Paid,4
3,2638,Neurology,CareOne,65167.44,NaN,Paid,24
4,8089,ER,HealthPlus,62661.81,NaN,Paid,7
5,19310,ICU,MediCareX,60510.22,NaN,Paid,2
6,24623,Neurology,SecureLife,58275.40,NaN,Paid,2
7,18189,Neurology,HealthPlus,57521.12,NaN,Paid,4
8,14702,Orthopedics,HealthPlus,56848.91,NaN,Paid,1
9,5131,Orthopedics,MediCareX,56736.52,NaN,Paid,12


### Financial insights
1. Total billed amount is led by MediCareX (134.6M) and CareOne (130.7M), so payer-management focus on the top providers will cover the largest revenue surface area.

2. Claim rejection rate is highest for SecureLife (15.69%), followed by MediCareX (15.25%), making these providers candidates for root-cause analysis (documentation gaps, coding mismatches, policy exclusions).

3. Payment delays are very similar across providers (~13 days on average), so cashflow improvements may depend more on reducing rejections and improving approvals than on provider-specific collections speed alone.

4. Revenue realization (approved/billed) is lowest in Orthopedics (~0.7685) and Cardiology (~0.7708), indicating comparatively higher leakage or down-coding/denial pressure in those departments.

### Revenue leakage flags
There are 190 high-billed (top decile) visits with approved amount missing or zero, which is a high-priority queue for finance ops because each case is large-value and may reflect denial, documentation issues, or data capture gaps.

In [23]:
dq_checks = {
    "Visits without a billing record": """
        SELECT COUNT(*) FROM visits v
        LEFT JOIN billing b ON b.visit_id = v.visit_id
        WHERE b.visit_id IS NULL
    """,
    "Billing records without a visit": """
        SELECT COUNT(*) FROM billing b
        LEFT JOIN visits v ON v.visit_id = b.visit_id
        WHERE v.visit_id IS NULL
    """,
    "Duplicate patient_id in patients": """
        SELECT COUNT(*) FROM (
            SELECT patient_id FROM patients
            GROUP BY patient_id HAVING COUNT(*) > 1
        )
    """,
    "Invalid length_of_stay_hours (NULL or < 0)": """
        SELECT COUNT(*) FROM visits
        WHERE length_of_stay_hours IS NULL OR length_of_stay_hours < 0
    """,
    "Missing or invalid payment_days": """
        SELECT COUNT(*) FROM billing
        WHERE payment_days IS NULL OR payment_days < 0
    """,
    "Missing approved_amount": """
        SELECT COUNT(*) FROM billing WHERE approved_amount IS NULL
    """,
    "Visits with no insurance info on patient": """
        SELECT COUNT(*) FROM visits v
        JOIN patients p ON p.patient_id = v.patient_id
        WHERE p.insurance_provider IS NULL OR TRIM(p.insurance_provider) = ''
    """,
}

print("=== Data Quality Report ===\n")
for check, sql in dq_checks.items():
    result = con.execute(sql).fetchone()[0]
    status = "PASS" if result == 0 else f" {result} records"
    print(f"{status}  |  {check}")

=== Data Quality Report ===

PASS  |  Visits without a billing record
PASS  |  Billing records without a visit
PASS  |  Duplicate patient_id in patients
PASS  |  Invalid length_of_stay_hours (NULL or < 0)
 790 records  |  Missing or invalid payment_days
 1318 records  |  Missing approved_amount
PASS  |  Visits with no insurance info on patient


In [25]:
q1.to_csv("phase1_top_departments_by_volume.csv", index=False)
q2.to_csv("phase1_top_departments_avg_los.csv", index=False)
q3.to_csv("phase1_pct_high_risk_by_dept.csv", index=False)
q6.to_csv("phase1_top_insurance_billed.csv", index=False)
q7.to_csv("phase1_insurance_rejection_rate.csv", index=False)
q9.to_csv("phase1_realization_ratio_by_dept.csv", index=False)
print("Phase 1 result CSVs saved.")

Phase 1 result CSVs saved.
